# Euclidean ShellForce Corr Model on Continuous ABP

This notebook trains a short-time ShellForce CNEEP model on continuous-space WCA active Brownian particles.  Particle centers are converted to a particle-size-aware center-count field.  The model uses Euclidean annuli and `learned_absolute` shell messages, which is better matched to the radial WCA interaction than the Chebyshev shells used for lattice fields.

The exact entropy production density is not used here.  Verification focuses on whether the learned EP-like signal has sensible trends against WCA potential energy, potential-energy release, minimum pair distance, and the WCA cutoff scale.

In [ ]:
import os
import sys

candidate_roots = [
    os.environ.get("CNEEP_V2_ROOT"),
    os.path.abspath(".."),
    os.path.abspath("."),
    "/home/user1/CNEEP_v2",
]

CNEEP_V2_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.exists(os.path.join(candidate, "data", "ABP", "core.py")):
        CNEEP_V2_ROOT = candidate
        break

if CNEEP_V2_ROOT is None:
    raise RuntimeError("Could not locate CNEEP_v2 root. Set CNEEP_V2_ROOT.")

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

print("CNEEP_v2 root:", CNEEP_V2_ROOT)

In [ ]:
from argparse import Namespace
from datetime import datetime
import math

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm import tqdm

from data.ABP import ABPParams, ContinuousABP, ABPFieldizer, recommended_center_grid_size
from utils.sampler import CartesianSeqSampler

## 1. Hyperparameters

In [ ]:
opt = Namespace()
opt.model_type = "ABPEuclideanShellForceCNEEP2D_Tanh"
opt.device = "cuda" if torch.cuda.is_available() else "cpu"

# alpha-NEEP objective
opt.alpha = -0.5
opt.beta = 1.0
opt.lam = 0.0
opt.threshold = 0.01

# data/model shape
opt.periodic = True
opt.positional = False
opt.n_components = 1
opt.seq_len = 2

# Euclidean ShellForce settings
opt.max_distance = 8
opt.include_k0 = True
opt.shell_center_mode = "relative_only"
opt.shell_force_bias = False
opt.shell_relative_mode = "learned_absolute"
opt.shell_weight_normalization = "none"
opt.shell_width = 1.0
opt.shell_offset = 0.0
opt.shell_force_activation = "tanh"

# training
opt.n_iter = 1000
opt.train_batch_size = 512
opt.test_batch_size = 512
opt.video_batch_size = 512
opt.lr = 1e-4
opt.wd = 1e-5
opt.input_scalar = 1
opt.loss_scalar = 1
opt.scalar = 1
opt.clip_norm = 1
opt.record_freq = 100
opt.seed = 5
opt.n_layer = 2
opt.n_channel = 32
opt.n_hidden = 2
opt.val_ratio = 0.25

abp_params = ABPParams(
    N=256,
    L=32.0,
    sigma=1.0,
    epsilon=1.0,
    mobility=1.0,
    force_clip=500.0,
    force_chunk_size=256,
    v0=16.0,
    Dr=1.0,
    Dt=0.01,
    dt=2.0e-4,
    seed=123,
    device=opt.device,
)

grid_size = max(48, recommended_center_grid_size(abp_params.L, abp_params.sigma))
fieldizer = ABPFieldizer(
    box_size=abp_params.L,
    grid_size=grid_size,
    particle_diameter=abp_params.sigma,
    mode="center",
    include_orientation=False,
    clip_occupancy=False,
)
opt.input_shape = (grid_size, grid_size)

# Simulation controls. Increase these after the smoke run looks healthy.
n_trajs = 8
n_trajs_test = 2
burn_in = 2_000
n_steps = 6_000
save_interval = 30
n_steps_test = 6_000
test_seed = 456
dt_saved = abp_params.dt * save_interval

torch.manual_seed(opt.seed)
np.random.seed(opt.seed)

result_folder = os.path.join(CNEEP_V2_ROOT, "results")
current_result_folder = os.path.join(
    result_folder, f"CorrABP-EuclideanShellForce-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}"
)
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, "model_parameter.pth.tar")
best_checkpoint_path = os.path.join(current_result_folder, "best_model_parameter.pth.tar")

print(f"Device: {opt.device}")
print(f"Results: {current_result_folder}")
print(f"ABP phi={abp_params.phi:.3f}, Pe={abp_params.Pe:.2f}, WCA rc/sigma={abp_params.rc / abp_params.sigma:.4f}")
print(f"Count field grid: {grid_size}x{grid_size}, dx={fieldizer.dx:.4f}, sigma_pixels={abp_params.sigma / fieldizer.dx:.3f}")
print(f"Euclidean shells: learned_absolute, max_distance={opt.max_distance}")

## 2. Simulate ABP trajectories and fieldize centers

In [ ]:
def fields_to_video(fields):
    # fields: [T, B, C, H, W] -> [B, T, H, W] for density-only C=1.
    fields = fields.float()
    if fields.shape[2] != 1:
        raise ValueError("This notebook is configured for n_components=1.")
    return fields[:, :, 0].permute(1, 0, 2, 3).contiguous()


print(f"[INFO] Generating TRAIN ABP trajectories B={n_trajs}")
sim_train = ContinuousABP(abp_params)
result_train = sim_train.simulate(
    B=n_trajs,
    burn_in=burn_in,
    n_steps=n_steps,
    save_interval=save_interval,
    fieldizer=fieldizer,
    show_progress=True,
)
train_states = fields_to_video(result_train["fields"])

print(f"[INFO] Generating TEST ABP trajectories B={n_trajs_test}")
sim_test = ContinuousABP(ABPParams(**{**abp_params.__dict__, "seed": test_seed}))
result_test = sim_test.simulate(
    B=n_trajs_test,
    burn_in=burn_in,
    n_steps=n_steps_test,
    save_interval=save_interval,
    fieldizer=fieldizer,
    show_progress=True,
)
test_states = fields_to_video(result_test["fields"])

print("Train density states:", train_states.shape)
print("Test density states: ", test_states.shape)
print("Final field diagnostics:", fieldizer.diagnostics_dict(result_train["positions"][-1].to(opt.device)))

## 3. Prepare tensors and normalization

In [ ]:
opt.M = train_states.shape[0]
opt.L = train_states.shape[1]
opt.M_test = test_states.shape[0]
opt.L_test = test_states.shape[1]

train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

train_video = train_states[:M_train_new].float().to(opt.device)
val_video = train_states[M_train_new:].float().to(opt.device)
test_video = test_states.float().to(opt.device)

mean = torch.mean(train_video, dim=(0, 1, 2, 3), keepdim=True)
std = torch.std(train_video, dim=(0, 1, 2, 3), keepdim=True).clamp_min(1e-6)
transform = lambda x: (x - mean.to(x.device)) * opt.input_scalar / std.to(x.device)

train_U = result_train["potential"].numpy().T
test_U = result_test["potential"].numpy().T
test_min_dist = result_test["min_distance"].numpy().T
test_times = result_test["times"].numpy()

print("Train video:", train_video.shape)
print("Val video:  ", val_video.shape)
print("Test video: ", test_video.shape)
print("Density mean:", float(mean.detach().cpu()))
print("Density std: ", float(std.detach().cpu()))
print("Mean WCA U train/test:", float(train_U.mean()), float(test_U.mean()))

## 4. Quick data sanity plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, t in enumerate([0, opt.L // 2, opt.L - 1]):
    im = axes[0, col].imshow(train_video[0, t].detach().cpu().numpy().T, origin="lower", cmap="viridis")
    axes[0, col].set_title(f"train count field t={t}")
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

axes[1, 0].plot(result_train["times"].numpy(), train_U.T, alpha=0.6)
axes[1, 0].set_title("WCA potential")
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("U")

axes[1, 1].plot(result_train["times"].numpy(), result_train["min_distance"].numpy() / abp_params.sigma, alpha=0.6)
axes[1, 1].axhline(1.0, color="k", linestyle="--", lw=1)
axes[1, 1].set_title("min distance / sigma")
axes[1, 1].set_xlabel("time")

axes[1, 2].hist(train_U.reshape(-1), bins=40, alpha=0.8)
axes[1, 2].set_title("WCA U distribution")
axes[1, 2].set_xlabel("U")

plt.tight_layout()
plt.show()

## 5. Build Euclidean ShellForce model

In [ ]:
from models.NEEP_ABP_ShellForce_2D import ABPEuclideanShellForceCNEEP2D, ABPEuclideanShellForceCNEEP2D_Tanh

if opt.model_type == "ABPEuclideanShellForceCNEEP2D_Tanh":
    model = ABPEuclideanShellForceCNEEP2D_Tanh(opt).to(opt.device)
elif opt.model_type == "ABPEuclideanShellForceCNEEP2D":
    model = ABPEuclideanShellForceCNEEP2D(opt).to(opt.device)
else:
    raise ValueError(f"Unknown model_type: {opt.model_type}")

optim = torch.optim.AdamW(model.parameters(), opt.lr, weight_decay=opt.wd)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print("annulus bounds in pixels:", model.shell_bounds())
print("annulus bounds in sigma units:", [(a * fieldizer.dx / abp_params.sigma, b * fieldizer.dx / abp_params.sigma) for a, b in model.shell_bounds()])

## 6. Train

In [ ]:
train_sampler = CartesianSeqSampler(
    M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device
)
val_sampler = CartesianSeqSampler(
    M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False
)

best_val_loss = float("inf")
smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None
history_train_loss = []
history_val_loss = []
history_iters = []

plt.ion()
fig, ax = plt.subplots(figsize=(8, 5))
line_train, = ax.plot([], [], label="Train Loss")
line_val, = ax.plot([], [], label="Val Loss")
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
ax.grid(True)
display_handle = display(fig, display_id=True)
plt.close(fig)

for it in tqdm(range(1, opt.n_iter + 1)):
    model.train()
    batch = next(train_sampler)
    b0 = batch[0].to(train_video.device)
    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]
    x = transform(torch.stack(slices, dim=1).float().to(opt.device))

    J_all = model(x) / opt.scalar
    ep_density = J_all.sum(dim=1)

    optim.zero_grad()
    if opt.alpha == 0:
        loss = (-ep_density + (torch.exp(-ep_density) - 1)).mean()
    else:
        loss = (
            -(torch.exp(opt.alpha * ep_density) - 1) / opt.alpha
            + (torch.exp(-(1 + opt.alpha) * ep_density) - 1) / (1 + opt.alpha)
        ).mean()

    (loss * opt.loss_scalar).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)
    optim.step()

    if it % opt.record_freq == 0 or it == 1:
        model.eval()
        val_loss_acc = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_sampler:
                vb0 = vb[0].to(val_video.device)
                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]
                vx = transform(torch.stack(vslices, dim=1).float().to(opt.device))
                vJ = model(vx) / opt.scalar
                v_ep_density = vJ.sum(dim=1)
                if opt.alpha == 0:
                    vloss = (-v_ep_density + (torch.exp(-v_ep_density) - 1)).sum().item()
                else:
                    vloss = (
                        -(torch.exp(opt.alpha * v_ep_density) - 1) / opt.alpha
                        + (torch.exp(-(1 + opt.alpha) * v_ep_density) - 1) / (1 + opt.alpha)
                    ).sum().item()
                val_loss_acc += vloss
                n_val += vx.shape[0]

        avg_val = val_loss_acc / max(n_val, 1)
        state = {
            "settings": opt.__dict__,
            "state_dict": model.state_dict(),
            "optimizer": optim.state_dict(),
            "iteration": it,
            "channel_mean": mean.detach().cpu(),
            "channel_std": std.detach().cpu(),
            "abp_params": abp_params.__dict__,
            "fieldizer": {
                "box_size": fieldizer.box_size,
                "grid_size": fieldizer.grid_size,
                "particle_diameter": fieldizer.particle_diameter,
                "mode": fieldizer.mode,
                "include_orientation": fieldizer.include_orientation,
                "clip_occupancy": fieldizer.clip_occupancy,
                "gaussian_sigma": fieldizer.gaussian_sigma,
                "gaussian_sigma_pixels": fieldizer.gaussian_sigma_pixels,
                "gaussian_truncate": fieldizer.gaussian_truncate,
                "gaussian_normalize": fieldizer.gaussian_normalize,
            },
            "dt_saved": dt_saved,
        }
        torch.save(state, current_checkpoint_path)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(state, best_checkpoint_path)

        smooth_train_loss = loss.item() if smooth_train_loss is None else smoothing * smooth_train_loss + (1 - smoothing) * loss.item()
        smooth_val_loss = avg_val if smooth_val_loss is None else smoothing * smooth_val_loss + (1 - smoothing) * avg_val
        history_train_loss.append(smooth_train_loss)
        history_val_loss.append(smooth_val_loss)
        history_iters.append(it)
        line_train.set_data(history_iters, history_train_loss)
        line_val.set_data(history_iters, history_val_loss)
        ax.relim()
        ax.autoscale_view()
        display_handle.update(fig)

print("Training finished.")
print(f"Best checkpoint: {best_checkpoint_path}")

## 7. Load best or final model

In [ ]:
load_best = True
checkpoint_path = best_checkpoint_path if load_best and os.path.exists(best_checkpoint_path) else current_checkpoint_path
checkpoint = torch.load(checkpoint_path, map_location=opt.device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Loaded {checkpoint_path}")
print(f"Iteration: {checkpoint.get('iteration', 'unknown')}")

## 8. Predict test-pair EP-like increments

In [ ]:
model.eval()
pred_increment = []
pred_shell_increment = []
pair_b = []
pair_t = []

test_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False
)

with torch.no_grad():
    for batch in tqdm(test_sampler):
        b0 = batch[0].to(test_video.device)
        t0 = batch[1][0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.stack(slices, dim=1).float().to(opt.device))
        J_all = model(x) / opt.scalar
        shell_inc = J_all.detach().cpu().numpy() * (grid_size ** 2)
        total_inc = shell_inc.sum(axis=1)
        pred_shell_increment.append(shell_inc)
        pred_increment.append(total_inc)
        pair_b.append(batch[0].detach().cpu().numpy())
        pair_t.append(batch[1][0].detach().cpu().numpy())

pred_increment = np.concatenate(pred_increment)
pred_shell_increment = np.concatenate(pred_shell_increment, axis=0)
pair_b = np.concatenate(pair_b)
pair_t = np.concatenate(pair_t)

U_t = test_U[pair_b, pair_t]
U_tp1 = test_U[pair_b, pair_t + 1]
dU_dt = (U_tp1 - U_t) / dt_saved
release_rate = -dU_dt
min_r = test_min_dist[pair_b, pair_t]
pred_rate = pred_increment / dt_saved
shell_rate = pred_shell_increment / dt_saved
pair_time = test_times[pair_t]

def corrcoef(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])

print(f"Pred mean rate: {pred_rate.mean():.6e}")
print(f"corr(pred, U):        {corrcoef(pred_rate, U_t):.4f}")
print(f"corr(pred, -dU/dt):   {corrcoef(pred_rate, release_rate):.4f}")
print(f"corr(pred, min_dist): {corrcoef(pred_rate, min_r):.4f}")

## 9. Trend plots against WCA observables

In [ ]:
def smooth(x, window=11):
    window = min(window, len(x))
    if window <= 1:
        return x
    return np.convolve(x, np.ones(window) / window, mode="same")

ens = 0
mask = pair_b == ens
order = np.argsort(pair_t[mask])
t_plot = pair_time[mask][order]
pred_plot = pred_rate[mask][order]
U_plot = U_t[mask][order]
rel_plot = release_rate[mask][order]
minr_plot = min_r[mask][order] / abp_params.sigma

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(t_plot, pred_plot, alpha=0.35)
axes[0].plot(t_plot, smooth(pred_plot), lw=2)
axes[0].set_ylabel("Pred EP rate")

axes[1].plot(t_plot, U_plot, alpha=0.35, color="tab:orange")
axes[1].plot(t_plot, smooth(U_plot), lw=2, color="tab:orange")
axes[1].set_ylabel("WCA U")

axes[2].plot(t_plot, rel_plot, alpha=0.35, color="tab:green")
axes[2].plot(t_plot, smooth(rel_plot), lw=2, color="tab:green")
axes[2].axhline(0, color="k", lw=1)
axes[2].set_ylabel("-dU/dt")

axes[3].plot(t_plot, minr_plot, alpha=0.35, color="tab:red")
axes[3].plot(t_plot, smooth(minr_plot), lw=2, color="tab:red")
axes[3].axhline(1.0, color="k", linestyle="--", lw=1)
axes[3].set_ylabel("min r/sigma")
axes[3].set_xlabel("time")

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_wca_trends.png"), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(U_t, pred_rate, s=8, alpha=0.35)
axes[0].set_xlabel("WCA U(t)")
axes[0].set_ylabel("Pred EP rate")
axes[0].set_title(f"corr={corrcoef(pred_rate, U_t):.3f}")

axes[1].scatter(release_rate, pred_rate, s=8, alpha=0.35)
axes[1].set_xlabel("-dU/dt")
axes[1].set_ylabel("Pred EP rate")
axes[1].set_title(f"corr={corrcoef(pred_rate, release_rate):.3f}")

axes[2].scatter(min_r / abp_params.sigma, pred_rate, s=8, alpha=0.35)
axes[2].axvline(1.0, color="k", linestyle="--", lw=1)
axes[2].set_xlabel("min pair distance / sigma")
axes[2].set_ylabel("Pred EP rate")
axes[2].set_title(f"corr={corrcoef(pred_rate, min_r):.3f}")

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_wca_scatter.png"), dpi=150)
plt.show()

## 10. Local predicted map vs center WCA-energy field

In [ ]:
def particle_wca_energy(pos_np, sim):
    pos = torch.as_tensor(pos_np, dtype=torch.float32, device=opt.device).unsqueeze(0)
    p = sim.params
    delta = pos[:, :, None, :] - pos[:, None, :, :]
    delta = delta - p.L * torch.round(delta / p.L)
    r2 = torch.sum(delta * delta, dim=-1)
    eye = torch.eye(pos.shape[1], device=pos.device, dtype=torch.bool).unsqueeze(0)
    mask = (r2 > 0) & (r2 < p.rc ** 2) & (~eye)
    r2_safe = torch.where(mask, r2, torch.ones_like(r2))
    sig2_over_r2 = (p.sigma ** 2) / r2_safe
    sig6 = sig2_over_r2 ** 3
    sig12 = sig6 ** 2
    u_pair = 4.0 * p.epsilon * (sig12 - sig6) + p.epsilon
    u_pair = torch.where(mask, u_pair, torch.zeros_like(u_pair))
    return (0.5 * u_pair.sum(dim=2))[0].detach().cpu().numpy()


def particle_values_to_center_field(pos_np, values, fieldizer):
    H = fieldizer.grid_size
    dx = fieldizer.dx
    idx = np.floor((pos_np % fieldizer.box_size) / dx).astype(np.int64) % H
    lin = idx[:, 0] * H + idx[:, 1]
    out = np.zeros(H * H, dtype=np.float64)
    np.add.at(out, lin, values)
    return out.reshape(H, H)


b = 0
t = int(opt.L_test * 0.6)
x = transform(torch.stack([test_video[b:b+1, t], test_video[b:b+1, t + 1]], dim=1).float().to(opt.device))
with torch.no_grad():
    maps = model(x, return_maps=True) / opt.scalar
pred_map_rate = maps[0].sum(dim=0).detach().cpu().numpy() * (grid_size ** 2) / dt_saved

pos_np = result_test["positions"][t, b].numpy()
u_particle = particle_wca_energy(pos_np, sim_test)
wca_field = particle_values_to_center_field(pos_np, u_particle, fieldizer)
occ = test_video[b, t].detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(occ.T, origin="lower", cmap="viridis")
axes[0].set_title("center count field")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

v = np.nanpercentile(np.abs(pred_map_rate), 99)
v = max(v, 1e-12)
im1 = axes[1].imshow(pred_map_rate.T, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
axes[1].set_title("predicted local EP rate")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(wca_field.T, origin="lower", cmap="magma")
axes[2].set_title("center WCA energy field")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_local_map_vs_wca.png"), dpi=150)
plt.show()

print("Frame WCA U from particles:", float(u_particle.sum()))
print("Frame WCA U from diagnostics:", float(test_U[b, t]))

## 11. Shell spectrum on Euclidean radial bins

In [ ]:
mean_shell = shell_rate.mean(axis=0)
stderr_shell = shell_rate.std(axis=0) / np.sqrt(max(len(shell_rate), 1))
bounds_px = model.shell_bounds()
centers_sigma = np.array([0.0 if b == 0 else 0.5 * (lo + hi) * fieldizer.dx / abp_params.sigma for b, (lo, hi) in enumerate(bounds_px)])
labels = [str(i) for i in range(len(bounds_px))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(centers_sigma, mean_shell, yerr=stderr_shell, width=0.55 * fieldizer.dx / abp_params.sigma, capsize=3, alpha=0.8)
axes[0].axvline(abp_params.rc / abp_params.sigma, color="k", linestyle="--", label="WCA cutoff")
axes[0].set_xlabel("annulus center radius / sigma")
axes[0].set_ylabel("mean predicted shell rate")
axes[0].set_title("Euclidean ShellForce spectrum")
axes[0].legend()

axes[1].plot(centers_sigma, np.cumsum(mean_shell), "o-")
axes[1].axvline(abp_params.rc / abp_params.sigma, color="k", linestyle="--", label="WCA cutoff")
axes[1].set_xlabel("annulus center radius / sigma")
axes[1].set_ylabel("cumulative predicted rate")
axes[1].set_title("Cumulative shell contribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_shell_spectrum.png"), dpi=150)
plt.show()

for idx, (lo, hi) in enumerate(bounds_px):
    print(
        f"branch {idx:02d}: pixels=({lo:.2f}, {hi:.2f}], "
        f"sigma_units=({lo * fieldizer.dx / abp_params.sigma:.3f}, {hi * fieldizer.dx / abp_params.sigma:.3f}], "
        f"mean_rate={mean_shell[idx]:.6e}"
    )

## 12. Inspect Euclidean annulus masks

In [ ]:
branches = [branch for branch in model.branches if branch.k > 0]
n_show = min(4, len(branches))
fig, axes = plt.subplots(1, n_show, figsize=(3.3 * n_show, 3.2))
if n_show == 1:
    axes = [axes]
for ax, branch in zip(axes, branches[:n_show]):
    offsets = branch.rel_proj.offsets.detach().cpu().numpy()
    ax.scatter(offsets[:, 1], offsets[:, 0], s=45)
    circle_outer = plt.Circle((0, 0), branch.r_outer, fill=False, color="k", linestyle="--")
    circle_inner = plt.Circle((0, 0), branch.r_inner, fill=False, color="gray", linestyle=":")
    ax.add_patch(circle_outer)
    ax.add_patch(circle_inner)
    ax.axhline(0, color="0.8", lw=1)
    ax.axvline(0, color="0.8", lw=1)
    ax.set_aspect("equal")
    ax.set_title(f"k={branch.k}: ({branch.r_inner:.1f},{branch.r_outer:.1f}] px")
plt.tight_layout()
plt.show()